# Network-Multimodal LightGCN for Yelp

This notebook follows the training/evaluation/checkpoint flow of `train_multimodal_lightgcn_spatiotemporal.py`, but uses **network modalities**:

- User-Item interaction graph (`review.json`)
- User-User social graph (`user.json` friends)
- Item-Item similarity graph (`business.json` categories)

Model: relation-aware LightGCN + rating-embedding branch + MLP scorer.

Outputs:
- `network_multimodal_lightgcn_state.pt`
- `network_run_summary.json`


In [ ]:
import os
import json
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class Config:
    data_dir: str = "."
    business_json: str = "yelp_academic_dataset_business.json"
    user_json: str = "yelp_academic_dataset_user.json"
    review_json: str = "yelp_academic_dataset_review.json"

    max_reviews: Optional[int] = None  # None = full dataset
    review_chunksize: int = 300000
    user_chunksize: int = 100000
    business_chunksize: int = 100000

    epochs: int = 12
    batch_size: int = 4096
    lr: float = 1e-3
    emb_dim: int = 64
    n_layers: int = 3
    star_emb_dim: int = 16
    mlp_hidden: int = 64
    alpha_social: float = 0.5
    alpha_item_sim: float = 0.5

    val_ratio: float = 0.1
    test_ratio: float = 0.1
    val_max_samples: int = 100000
    test_max_samples: int = 150000

    micro_batches_per_propagate: int = 32
    log_every: int = 50

    category_window: int = 15
    max_items_per_category: int = 400

    checkpoint_dir: str = "./checkpoints"
    checkpoint_name: str = "network_multimodal_lightgcn_state.pt"

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[info] device: {device}")
print(f"[info] full-data mode: {cfg.max_reviews is None}")


In [ ]:
def resolve_yelp_json_path(filename: str, data_dir: str = ".") -> str:
    candidates = [
        os.path.join(data_dir, filename),
        os.path.join(os.getcwd(), filename),
        os.path.join(os.getcwd(), "Yelp JSON", filename),
        os.path.join(os.getcwd(), "Yelp JSON", "yelp_dataset", filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return os.path.abspath(p)
    raise FileNotFoundError(f"Could not resolve {filename}. Checked: {candidates}")

def load_interactions(review_path: str, max_reviews: Optional[int], chunksize: int):
    usecols = ["user_id", "business_id", "stars"]
    uids, bids, stars = [], [], []
    loaded = 0

    for chunk in pd.read_json(review_path, lines=True, chunksize=chunksize):
        chunk = chunk[usecols].dropna(subset=["user_id", "business_id"])
        if max_reviews is not None:
            remain = max_reviews - loaded
            if remain <= 0:
                break
            chunk = chunk.iloc[:remain]

        uids.extend(chunk["user_id"].astype(str).tolist())
        bids.extend(chunk["business_id"].astype(str).tolist())
        s = pd.to_numeric(chunk["stars"], errors="coerce").fillna(3.0)
        stars.extend(np.round(s).clip(1, 5).astype(np.int64).tolist())

        loaded += len(chunk)
        if loaded % 1000000 == 0:
            print(f"[info] loaded reviews: {loaded:,}")

    df = pd.DataFrame({"user_id": uids, "business_id": bids, "stars": stars})
    u_codes, u_uniques = pd.factorize(df["user_id"], sort=False)
    i_codes, i_uniques = pd.factorize(df["business_id"], sort=False)
    u_map = {k: int(v) for v, k in enumerate(u_uniques.tolist())}
    i_map = {k: int(v) for v, k in enumerate(i_uniques.tolist())}

    u_idx = u_codes.astype(np.int64)
    i_idx = i_codes.astype(np.int64)
    star_idx = df["stars"].to_numpy(dtype=np.int64)
    return u_idx, i_idx, star_idx, u_map, i_map

def build_unique_bipartite_edges(user_idx: np.ndarray, item_idx: np.ndarray):
    pairs = np.stack([user_idx, item_idx], axis=1)
    uniq = np.unique(pairs, axis=0)
    edge_u = torch.from_numpy(uniq[:, 0].astype(np.int64))
    edge_i = torch.from_numpy(uniq[:, 1].astype(np.int64))

    num_u = int(user_idx.max()) + 1
    num_i = int(item_idx.max()) + 1
    deg_u = torch.bincount(edge_u, minlength=num_u).float().clamp(min=1.0)
    deg_i = torch.bincount(edge_i, minlength=num_i).float().clamp(min=1.0)
    norm = (deg_u[edge_u] * deg_i[edge_i]).pow(-0.5)
    return edge_u, edge_i, norm

def build_homogeneous_norm_edges(pairs: np.ndarray, num_nodes: int):
    if pairs.size == 0:
        return None, None, None

    pairs = pairs.astype(np.int64)
    rev = pairs[:, [1, 0]]
    all_pairs = np.concatenate([pairs, rev], axis=0)
    all_pairs = np.unique(all_pairs, axis=0)

    src = torch.from_numpy(all_pairs[:, 0])
    dst = torch.from_numpy(all_pairs[:, 1])
    deg = torch.bincount(src, minlength=num_nodes).float().clamp(min=1.0)
    norm = (deg[src] * deg[dst]).pow(-0.5)
    return src, dst, norm

def build_social_edges(user_path: str, user_to_idx: Dict[str, int], chunksize: int):
    edges = []
    active = set(user_to_idx.keys())

    for chunk in pd.read_json(user_path, lines=True, chunksize=chunksize):
        chunk = chunk[["user_id", "friends"]]
        chunk = chunk[chunk["user_id"].isin(active)]
        for row in chunk.itertuples(index=False):
            u = user_to_idx.get(str(row.user_id))
            if u is None:
                continue
            fs = str(row.friends)
            if not fs or fs == "None":
                continue
            for f in fs.split(","):
                f = f.strip()
                v = user_to_idx.get(f)
                if v is not None and v != u:
                    edges.append((u, v))

    if not edges:
        return np.empty((0, 2), dtype=np.int64)
    return np.array(edges, dtype=np.int64)

def build_item_similarity_edges(
    business_path: str,
    item_to_idx: Dict[str, int],
    chunksize: int,
    window: int,
    max_items_per_category: int,
):
    token_to_items: Dict[str, List[int]] = {}
    active = set(item_to_idx.keys())
    rng = np.random.default_rng(SEED)

    for chunk in pd.read_json(business_path, lines=True, chunksize=chunksize):
        cols = ["business_id", "categories"]
        chunk = chunk[cols]
        chunk = chunk[chunk["business_id"].isin(active)]
        for row in chunk.itertuples(index=False):
            bid = str(row.business_id)
            i = item_to_idx.get(bid)
            if i is None:
                continue
            cats = str(row.categories)
            if not cats or cats == "None":
                continue
            for c in cats.split(","):
                token = c.strip().lower()
                if token:
                    token_to_items.setdefault(token, []).append(i)

    sim_edges = set()
    for _, items in token_to_items.items():
        uniq = np.array(sorted(set(items)), dtype=np.int64)
        if uniq.size < 2:
            continue
        if uniq.size > max_items_per_category:
            uniq = rng.choice(uniq, size=max_items_per_category, replace=False)
            uniq.sort()
        m = uniq.size
        w = min(window, m - 1)
        for a in range(m):
            ia = int(uniq[a])
            for b in range(1, w + 1):
                ib = int(uniq[(a + b) % m])
                if ia != ib:
                    if ia < ib:
                        sim_edges.add((ia, ib))
                    else:
                        sim_edges.add((ib, ia))

    if not sim_edges:
        return np.empty((0, 2), dtype=np.int64)
    return np.array(sorted(sim_edges), dtype=np.int64)


In [ ]:
class NetworkMultimodalLightGCN(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        emb_dim: int,
        n_layers: int,
        star_emb_dim: int,
        mlp_hidden: int,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.star_emb = nn.Embedding(6, star_emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.xavier_uniform_(self.star_emb.weight)

        in_dim = emb_dim * 2 + star_emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, mlp_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(mlp_hidden, 1),
        )
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def propagate(
        self,
        edge_u,
        edge_i,
        norm_ui,
        edge_uu_src,
        edge_uu_dst,
        norm_uu,
        edge_ii_src,
        edge_ii_dst,
        norm_ii,
        alpha_social: float,
        alpha_item: float,
    ):
        eu = self.user_emb.weight
        ei = self.item_emb.weight
        all_u = [eu]
        all_i = [ei]

        for _ in range(self.n_layers):
            msg_i2u = norm_ui.unsqueeze(1) * ei[edge_i]
            agg_u = torch.zeros_like(eu)
            agg_u.index_add_(0, edge_u, msg_i2u)

            msg_u2i = norm_ui.unsqueeze(1) * eu[edge_u]
            agg_i = torch.zeros_like(ei)
            agg_i.index_add_(0, edge_i, msg_u2i)

            if edge_uu_src is not None:
                social_msg = norm_uu.unsqueeze(1) * eu[edge_uu_src]
                social_u = torch.zeros_like(eu)
                social_u.index_add_(0, edge_uu_dst, social_msg)
                agg_u = agg_u + alpha_social * social_u

            if edge_ii_src is not None:
                sim_msg = norm_ii.unsqueeze(1) * ei[edge_ii_src]
                sim_i = torch.zeros_like(ei)
                sim_i.index_add_(0, edge_ii_dst, sim_msg)
                agg_i = agg_i + alpha_item * sim_i

            eu, ei = agg_u, agg_i
            all_u.append(eu)
            all_i.append(ei)

        out_u = torch.stack(all_u, dim=0).mean(dim=0)
        out_i = torch.stack(all_i, dim=0).mean(dim=0)
        return out_u, out_i

    def forward_scores(self, eu, ei, u, i, star_idx):
        u_vec = eu[u]
        i_vec = ei[i]
        s_vec = self.star_emb(star_idx)
        x = torch.cat([u_vec, i_vec, s_vec], dim=-1)
        return (u_vec * i_vec).sum(dim=-1) + self.mlp(x).squeeze(-1)

@torch.no_grad()
def pairwise_eval(
    model,
    rows_np,
    u_idx_t,
    i_idx_t,
    star_t,
    num_items,
    max_samples,
    batch_size,
    prop_args,
):
    if rows_np.size == 0:
        return float("nan"), float("nan")

    n = int(min(max_samples, rows_np.size))
    pick = np.random.default_rng(SEED + 1).choice(rows_np, size=n, replace=False)
    rows = torch.from_numpy(pick.astype(np.int64)).to(device)

    model.eval()
    eu, ei = model.propagate(**prop_args)

    pos_all, neg_all = [], []
    for s in range(0, n, batch_size):
        sl = rows[s:s + batch_size]
        u = u_idx_t[sl]
        ip = i_idx_t[sl]
        st = star_t[sl]
        j = (ip + torch.randint(1, num_items, (sl.numel(),), device=device)) % num_items

        pos = model.forward_scores(eu, ei, u, ip, st)
        neg = model.forward_scores(eu, ei, u, j, st)
        pos_all.append(pos.detach().float().cpu().numpy())
        neg_all.append(neg.detach().float().cpu().numpy())

    pos = np.concatenate(pos_all)
    neg = np.concatenate(neg_all)
    pair_acc = float((pos > neg).mean())

    y_true = np.concatenate([np.ones_like(pos, dtype=np.int32), np.zeros_like(neg, dtype=np.int32)])
    y_score = np.concatenate([pos, neg])
    try:
        auc = float(roc_auc_score(y_true, y_score))
    except ValueError:
        auc = float("nan")
    return pair_acc, auc


In [ ]:
t0 = time.perf_counter()

review_path = resolve_yelp_json_path(cfg.review_json, cfg.data_dir)
user_path = resolve_yelp_json_path(cfg.user_json, cfg.data_dir)
business_path = resolve_yelp_json_path(cfg.business_json, cfg.data_dir)

print(f"[info] review: {review_path}")
print(f"[info] user: {user_path}")
print(f"[info] business: {business_path}")

u_idx, i_idx, stars, u_map, i_map = load_interactions(
    review_path=review_path,
    max_reviews=cfg.max_reviews,
    chunksize=cfg.review_chunksize,
)
num_users = len(u_map)
num_items = len(i_map)
n_int = u_idx.shape[0]
print(f"[info] interactions={n_int:,} users={num_users:,} items={num_items:,}")

rng_split = np.random.default_rng(SEED)
rnd = rng_split.random(n_int)
is_test = rnd < cfg.test_ratio
is_val = (rnd >= cfg.test_ratio) & (rnd < cfg.test_ratio + cfg.val_ratio)
train_rows_np = np.where(~(is_test | is_val))[0].astype(np.int64)
val_rows_np = np.where(is_val)[0].astype(np.int64)
test_rows_np = np.where(is_test)[0].astype(np.int64)

if train_rows_np.size == 0:
    raise RuntimeError("No training rows. Lower val/test ratio.")

edge_u, edge_i, norm_ui = build_unique_bipartite_edges(u_idx[train_rows_np], i_idx[train_rows_np])
print(f"[info] unique UI train edges: {edge_u.numel():,}")

social_pairs = build_social_edges(user_path, u_map, cfg.user_chunksize)
item_pairs = build_item_similarity_edges(
    business_path,
    i_map,
    cfg.business_chunksize,
    cfg.category_window,
    cfg.max_items_per_category,
)
print(f"[info] social pairs(raw): {social_pairs.shape[0]:,}")
print(f"[info] item-sim pairs(raw): {item_pairs.shape[0]:,}")

uu_src, uu_dst, norm_uu = build_homogeneous_norm_edges(social_pairs, num_users)
ii_src, ii_dst, norm_ii = build_homogeneous_norm_edges(item_pairs, num_items)

edge_u = edge_u.to(device)
edge_i = edge_i.to(device)
norm_ui = norm_ui.to(device)
uu_src = uu_src.to(device) if uu_src is not None else None
uu_dst = uu_dst.to(device) if uu_dst is not None else None
norm_uu = norm_uu.to(device) if norm_uu is not None else None
ii_src = ii_src.to(device) if ii_src is not None else None
ii_dst = ii_dst.to(device) if ii_dst is not None else None
norm_ii = norm_ii.to(device) if norm_ii is not None else None

u_idx_t = torch.from_numpy(u_idx).to(device)
i_idx_t = torch.from_numpy(i_idx).to(device)
star_t = torch.from_numpy(stars).to(device)
train_rows_t = torch.from_numpy(train_rows_np).to(device)

print(f"[time] data + graph build: {time.perf_counter() - t0:.1f}s")


In [ ]:
model = NetworkMultimodalLightGCN(
    num_users=num_users,
    num_items=num_items,
    emb_dim=cfg.emb_dim,
    n_layers=cfg.n_layers,
    star_emb_dim=cfg.star_emb_dim,
    mlp_hidden=cfg.mlp_hidden,
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

n_train = int(train_rows_np.size)
n_batches = (n_train + cfg.batch_size - 1) // cfg.batch_size
mb_per_prop = n_batches if cfg.micro_batches_per_propagate <= 0 else min(max(1, cfg.micro_batches_per_propagate), n_batches)
n_chunks = (n_batches + mb_per_prop - 1) // mb_per_prop

prop_args = {
    "edge_u": edge_u,
    "edge_i": edge_i,
    "norm_ui": norm_ui,
    "edge_uu_src": uu_src,
    "edge_uu_dst": uu_dst,
    "norm_uu": norm_uu,
    "edge_ii_src": ii_src,
    "edge_ii_dst": ii_dst,
    "norm_ii": norm_ii,
    "alpha_social": cfg.alpha_social,
    "alpha_item": cfg.alpha_item_sim,
}

print(f"[info] micro-batches per propagate: {mb_per_prop} | optimizer steps/epoch: {n_chunks}")

last_val_acc = float("nan")
last_val_auc = float("nan")
train_start = time.perf_counter()

for epoch in range(cfg.epochs):
    model.train()
    perm = torch.randperm(n_train, device=device)
    losses = []

    for chunk_start in range(0, n_batches, mb_per_prop):
        chunk_end = min(chunk_start + mb_per_prop, n_batches)
        chunk_len = chunk_end - chunk_start
        opt.zero_grad()

        eu, ei = model.propagate(**prop_args)

        for bi, b in enumerate(range(chunk_start, chunk_end)):
            st = b * cfg.batch_size
            ed = min(st + cfg.batch_size, n_train)
            idx = train_rows_t[perm[st:ed]]

            u = u_idx_t[idx]
            i_pos = i_idx_t[idx]
            star_b = star_t[idx]
            i_neg = (i_pos + torch.randint(1, num_items, (idx.numel(),), device=device)) % num_items

            s_pos = model.forward_scores(eu, ei, u, i_pos, star_b)
            s_neg = model.forward_scores(eu, ei, u, i_neg, star_b)
            loss_mb = -F.logsigmoid(s_pos - s_neg).mean()
            reg = (
                model.user_emb.weight[u].pow(2).mean()
                + model.item_emb.weight[i_pos].pow(2).mean()
                + model.item_emb.weight[i_neg].pow(2).mean()
            ) * 1e-4
            loss_mb = loss_mb + reg
            losses.append(float(loss_mb.detach().item()))

            is_last = bi == chunk_len - 1
            (loss_mb / float(chunk_len)).backward(retain_graph=not is_last)

            if cfg.log_every > 0 and (b + 1) % cfg.log_every == 0:
                print(
                    f"[epoch {epoch + 1}/{cfg.epochs}] batch {b + 1}/{n_batches} "
                    f"loss={np.mean(losses[-cfg.log_every:]):.4f}",
                    flush=True,
                )

        opt.step()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    mean_loss = float(np.mean(losses))
    print(f"[epoch {epoch + 1}/{cfg.epochs}] loss_avg={mean_loss:.4f}")

    if val_rows_np.size > 0:
        last_val_acc, last_val_auc = pairwise_eval(
            model=model,
            rows_np=val_rows_np,
            u_idx_t=u_idx_t,
            i_idx_t=i_idx_t,
            star_t=star_t,
            num_items=num_items,
            max_samples=cfg.val_max_samples,
            batch_size=cfg.batch_size,
            prop_args=prop_args,
        )
        print(f"[val] pairwise_acc={last_val_acc:.4f} roc_auc={last_val_auc:.4f}")

test_acc, test_auc = pairwise_eval(
    model=model,
    rows_np=test_rows_np,
    u_idx_t=u_idx_t,
    i_idx_t=i_idx_t,
    star_t=star_t,
    num_items=num_items,
    max_samples=cfg.test_max_samples,
    batch_size=cfg.batch_size,
    prop_args=prop_args,
)
print(f"[test] pairwise_acc={test_acc:.4f} roc_auc={test_auc:.4f}")

train_secs = time.perf_counter() - train_start
total_secs = time.perf_counter() - t0
print(f"[time] training: {train_secs:.1f}s | total: {total_secs:.1f}s")

os.makedirs(cfg.checkpoint_dir, exist_ok=True)
ckpt_path = os.path.join(cfg.checkpoint_dir, cfg.checkpoint_name)

payload = {
    "model_state_dict": model.state_dict(),
    "meta": {
        "num_users": num_users,
        "num_items": num_items,
        "emb_dim": cfg.emb_dim,
        "n_layers": cfg.n_layers,
        "star_emb_dim": cfg.star_emb_dim,
        "mlp_hidden": cfg.mlp_hidden,
        "alpha_social": cfg.alpha_social,
        "alpha_item_sim": cfg.alpha_item_sim,
        "epochs": cfg.epochs,
        "max_reviews": cfg.max_reviews,
        "device": str(device),
        "val_pairwise_accuracy": None if np.isnan(last_val_acc) else float(last_val_acc),
        "val_roc_auc": None if np.isnan(last_val_auc) else float(last_val_auc),
        "test_pairwise_accuracy": None if np.isnan(test_acc) else float(test_acc),
        "test_roc_auc": None if np.isnan(test_auc) else float(test_auc),
        "train_seconds": round(train_secs, 2),
        "total_seconds": round(total_secs, 2),
        "micro_batches_per_propagate": mb_per_prop,
        "optimizer_steps_per_epoch": n_chunks,
        "graph_stats": {
            "ui_unique_edges": int(edge_u.numel()),
            "social_edges_directed": 0 if uu_src is None else int(uu_src.numel()),
            "item_sim_edges_directed": 0 if ii_src is None else int(ii_src.numel()),
        },
    },
    "id_maps": {
        "user_to_idx": u_map,
        "business_to_idx": i_map,
    },
}
torch.save(payload, ckpt_path)
print(f"[saved] {os.path.abspath(ckpt_path)}")

summary_path = os.path.join(cfg.checkpoint_dir, "network_run_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "checkpoint": os.path.abspath(ckpt_path),
            "train_seconds": round(train_secs, 2),
            "total_seconds": round(total_secs, 2),
            "val_pairwise_accuracy": None if np.isnan(last_val_acc) else float(last_val_acc),
            "val_roc_auc": None if np.isnan(last_val_auc) else float(last_val_auc),
            "test_pairwise_accuracy": None if np.isnan(test_acc) else float(test_acc),
            "test_roc_auc": None if np.isnan(test_auc) else float(test_auc),
            "config": vars(cfg),
        },
        f,
        indent=2,
    )
print(f"[saved] {os.path.abspath(summary_path)}")
